# 🧬 TFB Prediction Tutorial 1/4: From Biological Questions to Data Pipelines

Welcome to the first tutorial in our four-part series. This guide focuses on the foundational and most critical step of any computational biology project: **understanding and preparing your data**.

Before we write any code, we must first understand the landscape of biological data and how to frame biological questions as machine learning tasks.

### 1. The Language of Life: DNA and RNA

At its core, genomics is the study of sequences. The primary types are:
- **DNA (Deoxyribonucleic acid)**: The blueprint of life, composed of four bases (A, T, C, G). It carries the genetic instructions for the development, functioning, growth, and reproduction of all known organisms.
- **RNA (Ribonucleic acid)**: Often a messenger, RNA (composed of A, U, C, G) plays a crucial role in translating the instructions from DNA into proteins. It can also have structural and regulatory roles.

These sequences are not random strings, and they contain complex patterns, grammar, and syntax. Our goal is to teach a machine to read and understand this "language of life."

### 2. Framing Biological Questions as ML Tasks

A biological question must be translated into a well-defined machine learning task. Here are the most common types:

| Task Type | Biological Question | Example |
| :--- | :--- | :--- |
| **Sequence Classification** | Does this DNA sequence have a specific property? | Is this sequence a promoter region? (Yes/No) |
| **Multi-Label Classification**| What properties does this DNA sequence have? | Which of these 100 transcription factors will bind to this sequence? |
| **Sequence Regression** | What is the quantitative value associated with this sequence? | How efficiently will this mRNA be translated into a protein? (Predict a score) |
| **Token Classification** | What is the function of *each base* in this sequence? | Identify the start and end of a gene within a long DNA strand. |
| **Sequence-to-Sequence** | Can we translate this sequence into another? | Given a DNA coding sequence, predict the resulting protein sequence. |

```mermaid
graph TD
    subgraph "Mapping Problems to Tasks"
        A["Biological Question<br/>(e.g., Where do proteins bind?)"] --> B["ML Task<br/>(e.g., Multi-Label Sequence Classification)"];
        B --> C["Model Choice<br/>(e.g., OmniModelForMultiLabelSequenceClassification)"];
        C --> D["Data Representation<br/>(e.g., DNA sequence as input, multi-hot vector as output)"];
    end
```

### 3. The OmniGenBench Toolbox: Available Models for Every Task

`OmniGenBench` provides a suite of pre-configured models, each designed for a specific task. This saves you from having to build them from scratch.

| Task | OmniGenBench Model | When to Use It |
| :--- | :--- | :--- |
| Single-Label Sequence Classification | `OmniModelForSequenceClassification` | Predicting one label for the whole sequence. |
| **Multi-Label Sequence Classification** | `OmniModelForMultiLabelSequenceClassification` | **(Our focus)** Predicting multiple labels for the whole sequence. |
| Sequence Regression | `OmniModelForSequenceRegression` | Predicting a single continuous value for the sequence. |
| Token Classification | `OmniModelForTokenClassification` | Predicting a label for each token (base) in the sequence. |
| Token Regression | `OmniModelForTokenRegression` | Predicting a continuous value for each token. |
| Sequence-to-Sequence | `OmniModelForSeq2Seq` | Generating an output sequence from an input sequence. |

By understanding this mapping, you can quickly select the right tool for your biological problem.

### 4. Our Task: Why TFB is Multi-Label Sequence Classification

Now, let's apply this to our tutorial's goal: **Transcription Factor Binding (TFB) Prediction**.

- **It's a Sequence task**: Our input is a DNA sequence.
- **It's a Classification task**: For each transcription factor, we are asking a "Yes/No" question: does it bind?
- **It's a Multi-Label task**: A single DNA sequence can be a binding site for *multiple* different TFs simultaneously. We aren't just picking one from a list; we are identifying all potential binders.

Therefore, the correct framing is **Multi-Label Sequence Classification**, and the right tool from our toolbox is `OmniModelForMultiLabelSequenceClassification`.

With this clear understanding, we can now proceed to prepare our data specifically for this task.

---

## 🛠️ Step-by-Step Guide: Preparing the DeepSEA Dataset

Now we move from theory to practice. This section will guide you through the hands-on process of preparing the data.

### 1.1: Environment Setup

First, let's install the required Python packages. `omnigenbench` is our core library, `transformers` provides the underlying model architecture, and the other packages are utilities for our workflow.

In [ ]:
!pip install -U numpy transformers omnigenbench autocuda findfile requests

Next, we import the libraries we just installed. This gives us the tools for data processing, deep learning, and interacting with the operating system.

A key part of this setup is determining the best available hardware for training. Our script will automatically prioritize a **CUDA-enabled GPU** if one is available, as this can accelerate training by 10-100x compared to a CPU.

In [ ]:
import os
import json
import torch
import numpy as np
import requests
import zipfile
from typing import Optional, List, Dict, Any

from omnigenbench import (
    OmniDataset,
)

from transformers import AutoTokenizer
import autocuda
import findfile

from omnigenbench import (
    OmniTokenizer,
    OmniForSequenceClassification,
    OmniDatasetForSequenceClassification,
)

DEVICE = autocuda.auto_cuda()
torch.manual_seed(42)
np.random.seed(42)

print(f"🚀 Current device: {DEVICE}")
print("📚 All libraries imported successfully!")

### 1.2: Global Configuration

To make our tutorial easy to modify and understand, we'll centralize all important parameters in this section. This is a best practice that makes experiments more reproducible.

In [ ]:
# Configuration for TFB Prediction
class TFBConfig:
    # Dataset
    DATASET_NAME = "yangheng/deepsea_tfb_prediction"
    LOCAL_CACHE_DIR = "deepsea_tfb_prediction"
    
    # Model
    MODEL_NAME = "yangheng/OmniGenome-52M"
    MAX_LENGTH = 512
    BATCH_SIZE = 16
    
    # Quick testing configuration (use smaller subset)
    NUM_TF_LABELS = 10  # Use 10 TFs for quick testing (out of 919 total)
    MAX_EXAMPLES = 1000  # Use 1000 samples for quick testing
    
    # For full training, uncomment these:
    # NUM_TF_LABELS = 919  # All transcription factors
    # MAX_EXAMPLES = None  # All samples

config = TFBConfig()

print("⚙️ TFB Prediction Configuration:")
print(f"  📊 Dataset: {config.DATASET_NAME}")
print(f"  🧬 Model: {config.MODEL_NAME}")
print(f"  📏 Sequence length: {config.MAX_LENGTH}")
print(f"  🎯 TF labels: {config.NUM_TF_LABELS}")
print(f"  🔢 Sample limit: {config.MAX_EXAMPLES if config.MAX_EXAMPLES else 'All'}")
print("✅ Configuration ready!")

### 1.3: Data Acquisition

With our environment configured, it's time to download the DeepSEA dataset. The function below automates this process by:
1.  Checking if the data already exists.
2.  Downloading the dataset from the specified URL if needed.
3.  Extracting the files.
4.  Cleaning up the temporary zip file.

This ensures we have the `train.jsonl`, `valid.jsonl`, and `test.jsonl` files ready for the next stage.

In [ ]:
# Load tokenizer

print("🔄 Loading tokenizer...")
tokenizer = OmniTokenizer.from_pretrained(config.MODEL_NAME, trust_remote_code=True)
print("✅ Tokenizer loaded!")

# Load TFB dataset using enhanced OmniDataset
print("Loading DeepSEA TFB dataset with automatic download...")
datasets = OmniDatasetForSequenceClassification.from_huggingface(
    dataset_name=config.DATASET_NAME,
    tokenizer=tokenizer,
    max_length=config.MAX_LENGTH,
    cache_dir=config.LOCAL_CACHE_DIR
)

# Apply subset filtering if configured
if config.MAX_EXAMPLES or config.NUM_TF_LABELS < 919:
    print(f"⚙️ Applying configuration filters...")
    
    # This would be applied during dataset creation in a real scenario
    # For demonstration, we show the full dataset size
    print(f"  Target TF labels: {config.NUM_TF_LABELS}/919")
    print(f"  Sample limit: {config.MAX_EXAMPLES if config.MAX_EXAMPLES else 'All'}")

print("✅ TFB dataset preparation complete!")

### 1.4: Custom Dataset and Data Loaders
Now that we have the data files, we need a way to load them into our model efficiently. We'll do this in two parts:

For most of the classification and regression tasks, the dataset have been integrated in OmniGenBench, e.g.,  

| Dataset Class | Task Type | Description |
| :--- | :--- | :--- |
| `OmniDatasetForSequenceClassification` | Sequence Classification | Tasks for classifying the entire sequence into one category (e.g., promoter vs. non-promoter). |
| `OmniDatasetForSequenceRegression` | Sequence Regression | Tasks for predicting a single continuous value for the entire sequence (e.g., translation efficiency). |
| `OmniDatasetForTokenClassification` | Token (Base) Classification | Tasks for assigning a label to each token (base) in the sequence (e.g., identifying splice sites). |
| `OmniDatasetForTokenRegression` | Token (Base) Regression | Tasks for predicting a continuous value for each token (base) in the sequence. |

To demonstrate how to customize a dataset, we here define a dataset for the deepsea dataset in the following cell.

#### A. The `DeepSEADataset` Class
This custom class acts as a bridge between our raw `.jsonl` files and the PyTorch ecosystem. It inherits from `OmniDataset` and tells our framework how to process each data entry. Specifically, it:
1.  **Processes a DNA sequence and its labels** via **prepare_input()**.
2.  **Truncates or pads the sequence** to a fixed length (`MAX_LENGTH`). This is crucial because language models require inputs of a consistent size.
3.  **Selects the specific TF labels** we want to train on.
4.  **Tokenizes the sequence**, converting the string of "A, C, G, T" into numerical tokens that the model can understand.
5.  **Formats the output** as PyTorch tensors.

#### B. Creating DataLoaders
Once the `DeepSEADataset` is defined, we use PyTorch's `DataLoader` to create an efficient pipeline. The `DataLoader` is responsible for:
1.  **Batching**: Grouping individual samples into batches (`BATCH_SIZE`).
2.  **Shuffling**: Randomly shuffling the training data each epoch to improve generalization.
3.  **Parallelism**: Loading data in the background so it's ready for the model when needed.


In [ ]:
class DeepSEADataset(OmniDataset):
    def __init__(
        self,
        data_source: str,
        tokenizer,
        max_length: Optional[int] = None,
        label_indices: Optional[List[int]] = None,
        **kwargs
    ):
        super().__init__(data_source, tokenizer, max_length, **kwargs)
        self.label_indices = label_indices

        for key, value in kwargs.items():
            self.metadata[key] = value

    def prepare_input(self, instance: Dict[str, Any], **kwargs) -> Dict[str, torch.Tensor]:
        """
        This is the only method you need to customize for your dataset. 
        It takes a single data instance (a dictionary) and processes 
        it to return a dictionary of torch tensors ready for model input.
        Imagine you have a dataset where each instance is a dictionary like:
        {
            "sequence": "ACGTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGC...",
            "label": [0, 1, 0, 0, 1, ..., 0],  # Multi-hot encoded labels for TF binding
            "auxiliary_info": "some additional info",  # e.g., matrix or file path to some extra data
        },
        You can access the sequence and labels using instance['sequence'] and instance['label'].
        """
        def truncate_sequence(seq: str, max_len: Optional[int]) -> str:
            if max_len is None:
                return seq
            if len(seq) == max_len:
                return seq
            elif len(seq) > max_len:
                start_idx = (len(seq) - max_len) // 2
                return seq[start_idx:start_idx + max_len]
            else:
                return seq + ("N" * (max_len - len(seq)))

        sequence = instance.get('sequence') or instance.get('seq')
        labels = instance.get('label', None) if 'label' in instance else instance.get('labels', None)

        tokenized_inputs = self.tokenizer(
            truncate_sequence(sequence, self.max_length),
            padding="do_not_pad",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
            **kwargs
        )

        labels_tensor = None
        if labels is not None:
            labels_tensor = torch.tensor(labels, dtype=torch.float32)
            if hasattr(self, 'label_indices') and self.label_indices is not None:
                labels_tensor = labels_tensor[torch.tensor(self.label_indices, dtype=torch.long)]

        tokenized_inputs["labels"] = labels_tensor

        for key in tokenized_inputs:
            if isinstance(tokenized_inputs[key], torch.Tensor) and tokenized_inputs[key].ndim > 1:
                tokenized_inputs[key] = tokenized_inputs[key].squeeze(0)

        return tokenized_inputs
    
print("📝 Custom dataset class definition completed!")


# Initialize the tokenizer, which is needed for the dataset.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("🏗️ Creating datasets...")
train_dataset = DeepSEADataset(
    data_source=train_file,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
    max_examples=MAX_EXAMPLES,
    label_indices=LABEL_INDICES,
)
print(f"  📊 Training dataset size: {len(train_dataset)}")

valid_dataset = DeepSEADataset(
    data_source=valid_file,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
    max_examples=MAX_EXAMPLES,
    label_indices=LABEL_INDICES,
)
print(f"  📊 Validation dataset size: {len(valid_dataset)}")

test_dataset = DeepSEADataset(
    data_source=test_file,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
    max_examples=MAX_EXAMPLES,
    label_indices=LABEL_INDICES,
)
print(f"  📊 Test dataset size: {len(test_dataset)}")

### 1.5 Create Dataloaders
<!-- explain what are dataloaders and why we need dataloaders -->
Dataloaders are a crucial component in the training of machine learning models, especially when dealing with large datasets. They provide an efficient way to load and preprocess data in batches, which is essential for training models on GPUs. By using dataloaders, we can:

1. **Batching**: Dataloaders allow us to group multiple samples into batches, which helps in speeding up the training process by utilizing parallel processing capabilities of modern hardware.

2. **Shuffling**: They can shuffle the data at every epoch, which helps in reducing overfitting and ensures that the model generalizes well.

3. **Parallel Data Loading**: Dataloaders can load data in parallel using multiple workers, which significantly reduces the time spent waiting for data to be loaded.

4. **Dynamic Padding**: For NLP tasks, dataloaders can handle variable-length sequences by applying dynamic padding, ensuring that all sequences in a batch are of the same length.

In summary, dataloaders streamline the data feeding process during training, making it more efficient and effective.

In [ ]:
print("\n🔄 Creating data loaders...")
train_loader = torch.utils.data.DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    pin_memory=True if DEVICE.type == 'cuda' else False, 
    num_workers=0
)
print(f"  🚂 Training loader: {len(train_loader)} batches")

valid_loader = torch.utils.data.DataLoader(
    valid_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,
    pin_memory=True if DEVICE.type == 'cuda' else False, 
    num_workers=0
)
print(f"  ✅ Validation loader: {len(valid_loader)} batches")

test_loader = torch.utils.data.DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,
    pin_memory=True if DEVICE.type == 'cuda' else False, 
    num_workers=0
)
print(f"  🧪 Test loader: {len(test_loader)} batches")

print("\n🎉 Data pipeline construction completed!")

### Summary and Next Steps

Congratulations! You have successfully built a complete data preparation pipeline. You've learned not only *how* to code a data pipeline but also *why* it's structured the way it is, starting from the biological question itself.

Your data is now ready to be used for training a model.

In the next tutorial, **[2/4: Model Initialization](./02_model_initialization.ipynb)**, we will take these `DataLoaders` and learn how to set up a pre-trained Genomic Foundation Model for our TFB prediction task.